# VeriTex AI image detector — Colab training

This notebook fine-tunes the pretrained ViT image classifier on a balanced subset of OpenFake, evaluates the selected checkpoint, and saves outputs to Google Drive. It is configured for a Colab TPU.

Before running: choose **Runtime → Change runtime type → TPU**. Keep `per_class=5000` for the planned training run; reduce it if the session is too short. The first TPU steps may take extra time for compilation.

In [ ]:
# Upload the project ZIP from your computer, then extract it.
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Upload veritex-training.zip')
project_zip = zip_names[0]
!mkdir -p /content/veritex-project
!unzip -oq "{project_zip}" -d /content/veritex-project
!find /content/veritex-project -maxdepth 3 -name train.py -print

In [ ]:
# Locate the training folder and install its Python dependencies.
from pathlib import Path
training_dirs = list(Path('/content/veritex-project').rglob('train.py'))
if not training_dirs:
    raise FileNotFoundError('train.py was not found in the uploaded ZIP')
TRAIN_DIR = training_dirs[0].parent
%cd $TRAIN_DIR
!pip install -r requirements.txt

In [ ]:
# Confirm native TorchTPU works on this Colab TPU runtime.
import torch
import torch_tpu
print('PyTorch:', torch.__version__)
print('Current accelerator:', torch.accelerator.current_accelerator())
x = torch.ones((2, 2), device='tpu')
print('TPU check:', (x @ x).cpu().tolist())


In [ ]:
# Mount Drive so checkpoints survive the end of the Colab session.
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/veritex-data'
MODEL_DIR = '/content/drive/MyDrive/veritex-model'
print('Model checkpoints will be saved to:', MODEL_DIR)

In [ ]:
# Fine-tune all ViT layers on 5,000 real and 5,000 fake training images.
# The script prepares/resumes the balanced subset, then validates each epoch.
!python train.py \
  --device tpu \
  --data-dir "$DATA_DIR" \
  --output-dir "$MODEL_DIR" \
  --per-class 5000 \
  --epochs 3 \
  --batch-size 8 \
  --grad-accumulation 2

## Evaluate

Run one evaluation at a time to keep the TPU session manageable. The sample cap is 500 per class (1,000 images total). Remove `--max-per-class 500` later for a larger report.

In [ ]:
# Start with the held-out core test set.
!python evaluate.py \
  --device tpu \
  --model-dir "$MODEL_DIR" \
  --config core \
  --split test \
  --max-per-class 500 \
  --output /content/drive/MyDrive/veritex-core-test.json

In [ ]:
# Optional: separate in-the-wild Reddit evaluation (also 500 per class).
!python evaluate.py \
  --device tpu \
  --model-dir "$MODEL_DIR" \
  --config reddit \
  --split test \
  --max-per-class 500 \
  --output /content/drive/MyDrive/veritex-reddit-test.json